### 🔧 Basic Setup

This cell imports all required Python libraries and sets the working directory to the main AISCO project folder.

**What happens here:**
- Imports essential packages (`os`, `pandas`, `trimesh`)
- Imports the project’s simulation functions (`simulate_stl`, `run_parallel_simulations`)
- Ensures the working directory is set to the project root (`AISCO-code`)

> 💡 All relative file paths used later in the notebook will be interpreted from this root directory.
> 
> ⚠️ If you run this notebook on another system, check that the path in `os.chdir()` points to your `AISCO-code` folder.

In [48]:
# --- Basic setup ---
import os
import sys
import pandas as pd
import trimesh

# Set working directory to project root (AISCO-code)
project_root = "/Users/leonlinsi/Documents/ETH_Zürich/ETH_general/aCentauri/AIsco Project/AISCO-code"
os.chdir(project_root)

# Add project root to Python module search path
if project_root not in sys.path:
    sys.path.append(project_root)

# Import your simulation functions
from AISCO.subprojects.WP3_Solarestimator.simulate_stl import (
    run_parallel_simulations,
    simulate_stl,
    process_stl_file
)

print("✅ Working directory set to:", os.getcwd())

✅ Working directory set to: /Users/leonlinsi/Documents/ETH_Zürich/ETH_general/aCentauri/AIsco Project/AISCO-code


### 🧩 Input Parameters

In this cell, you define all relevant input paths and simulation settings.

**Parameter overview:**
- `STL_FOLDER`: Folder containing all `.stl` models to be simulated.  
- `ROUTE_CSV`: Path to the CSV file containing geographic and time data (latitude, longitude, altitude, timestamp).  
- `OUTPUT_CSV`: Path to store the simulation results table.  
- `MAX_AREA`: Maximum solar area in square meters to be considered.  
- `N_JOBS`: Number of CPU cores to use in parallel. (`-1` = use all available cores)

> 💡 Tip: If your computer gets hot or noisy, reduce `N_JOBS` to `2` or `4`.

In [49]:
# --- User Inputs ---
# Folder containing STL files
STL_FOLDER = "AISCO/subprojects/WP3_Solarestimator/data/stl_models/outer_silvretta_shells"

# File to be simulated (only for Single Simulation (with/without saving))
# !Attention!: the file must be in the STL_FOLDER
FILENAME = "design_iter0_outer.stl"

# Path to CSV with route
ROUTE_CSV = "AISCO/subprojects/WP3_Solarestimator/data/race_routes/points.csv"

# Output path for results
OUTPUT_CSV = "data/simulation_results.csv"

# Maximum solar area (m²)
MAX_AREA = 6.0

# Number of parallel cores (-1 = all)
N_JOBS = 4

### 1. Single Simulation (without saving)
With the following cells, you can calculate the PV index and the selected solar area in m³ for a single mesh without saving the results in the `data` folder. If this is your intention, execute the cells below.

In [51]:
performance_index, selected_faces = simulate_stl(
    path_csv = ROUTE_CSV,
    mesh_path = STL_FOLDER+"/"+FILENAME,
)
print("Performance index:",performance_index)
print("used area:", round((sum(selected_faces.facets_area)/10e5),3),"m2")

KeyboardInterrupt: 

### 🧭 Visualize a Solar Deck for Single Simulation (without saving)

This cell allows you to visualize ***the simulated vehicle calculated above** and its selected solar deck overlay.

**Instructions:**
1. Select the correct `FILENAME` (must be in `STL_FOLDER`) for the STL file to be simulated.  
2. The full vehicle is shown in **green**.  
3. The chosen solar deck is shown in **transparent red**.  


In [42]:
from AISCO.subprojects.WP3_Solarestimator.read_mesh import sky_face

selected_faces.visual.face_colors = [255, 0, 0, 150]

mesh=trimesh.load_mesh(STL_FOLDER+"/"+FILENAME)
top_mesh = sky_face(mesh)
top_mesh.visual.face_colors = [0, 255, 0, 150] 

scene = trimesh.Scene([top_mesh, selected_faces])
scene.show(notebook=True)

### 2. Single Simulation (with saving)
With the following cells, you can calculate the PV index and the selected solar area in m³ for a single mesh and save the selected solar deck in `data/simulation_results`. If this is your intention, execute the cells below.

In [ ]:
# --- Run all simulations ---
dic_results = process_stl_file(
    stl_folder = STL_FOLDER,
    filename = FILENAME,
    path_csv = ROUTE_CSV
)

print("✅ Simulation completed.")
print(dic_results)

✅ Simulation completed.
{'filename': 'design_iter0_outer.stl', 'performance_index': np.float64(0.9645639878183199), 'duration_s': 181.14, 'output_file': 'AISCO/subprojects/WP3_Solarestimator/data/selected_solardecks/solardeck_design_iter0_outer.stl'}


### 3. Parallel Simulation for multiple STL (with saving)
With the following cells, you can calculate the PV index and the selected solar area in m³ for **multiple** meshes and saves the selected solar decks in `data/simulation_results`. If this is your intention, execute the cells below.

> 💡 Tip: If your computer gets hot or noisy, reduce `N_JOBS` in the input parameters to `2` or `4`.

##### 🚀 Run All Simulations

This cell starts the batch simulation.  
Each STL file is processed, simulated, and its solar performance index is calculated.

**What happens:**
- Finds all `.stl` files in the specified folder.  
- Runs simulations in parallel using `joblib`.  
- Saves the results as `simulation_results.csv`.  
- Exports each optimized solar deck to `data/selected_solardecks/`.

> ⏱️ Simulation time depends on the number of models and CPU performance.

In [ ]:
# --- Run all simulations ---
df_results = run_parallel_simulations(
    stl_folder=STL_FOLDER,
    path_csv=ROUTE_CSV,
    output_csv=OUTPUT_CSV,
    max_area=MAX_AREA,
    n_jobs=N_JOBS
)

print("✅ Simulation completed.")
display(df_results)

### 🧭 Visualize a Solar Deck (Manual Selection for saved solardecks)

This cell allows you to visualize one of the simulated vehicles and its selected solar deck overlay. **Here you can just use the visualization if you simulates your STL/STLs "with saving"**.

**Instructions:**
1. Choose an STL filename that appears in the results table.  
2. The full vehicle is shown in **green**.  
3. The chosen solar deck is shown in **transparent red**.  

> 💡 Works interactively inside the notebook via `scene.show(notebook=True)`.

In [ ]:
from ipywidgets import interact
import pyglet


def show_solardeck(filename):
    stl_path = os.path.join(STL_FOLDER, filename)
    solardeck_path = os.path.join(
    "AISCO", "subprojects", "WP3_Solarestimator", "data", "selected_solardecks",
    f"solardeck_{os.path.splitext(filename)[0]}.stl"
    
)
    top_mesh = trimesh.load(stl_path)
    selected_faces = trimesh.load(solardeck_path)

    selected_faces.visual.face_colors = [255, 0, 0, 150]
    top_mesh.visual.face_colors = [0, 255, 0, 150]

    scene = trimesh.Scene([top_mesh, selected_faces])
    return scene.show(viewer='jupyter')

# Dropdown menu with all STL files
interact(show_solardeck, filename=os.listdir(STL_FOLDER));

interactive(children=(Dropdown(description='filename', options=('design_iter179_outer.stl', 'design_iter103_ou…